In [19]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [29]:
FILES = {
    "Rule-based": "synthetic_students.csv",
    "Seed-based": "my_synthetic_students_data.csv",
}
OUT_FOLDER = "results"        
RANDOM_SEED = 42

In [31]:
COLUMNS = ["ID", "Age", "Study_hrs", "Attendance%", "Total_screen_time",
           "Sleep_hrs", "Internet_quality", "exam_score", "Result"]
NUMERIC = ["Age", "Study_hrs", "Attendance%", "Total_screen_time", "Sleep_hrs"]
CATEGORY = ["Internet_quality"]
FEATURES = NUMERIC + CATEGORY   # exam_score is NOT used (it would leak the answer)

out = Path(OUT_FOLDER)
out.mkdir(exist_ok=True)

In [33]:
def load_data(path):
    df = pd.read_csv(path)
    lookup = {c.lower(): c for c in COLUMNS}
    df.columns = [lookup.get(c.strip().replace(" ", "_").lower(), c.strip())
                  for c in df.columns]
    q = df["Internet_quality"]
    if pd.api.types.is_numeric_dtype(q):
        df["Internet_quality"] = pd.cut(q, [0, 2, 3, 5],
                                        labels=["Poor", "Average", "Good"]).astype(str)
    else:
        df["Internet_quality"] = q.astype(str).str.strip().str.title()
    df["target"] = (df["Result"].astype(str).str.strip().str.title() == "Pass").astype(int)
    return df

In [35]:
def make_model(kind):
    prep = ColumnTransformer([
        ("num", StandardScaler(), NUMERIC),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORY),
    ])
    clf = (LogisticRegression(max_iter=1000) if kind == "logistic"
           else RandomForestClassifier(n_estimators=200, random_state=RANDOM_SEED))
    return Pipeline([("prep", prep), ("clf", clf)])


data = {name: load_data(path) for name, path in FILES.items()}
names = list(data)

In [37]:
quality = pd.DataFrame({
    name: {
        "Rows": len(df),
        "Missing values": int(df.isna().sum().sum()),
        "Duplicate IDs": int(df["ID"].duplicated().sum()),
        "Duplicate rows (ignoring ID)": int(df.drop(columns=["ID"]).duplicated().sum()),
        "Result inconsistent with 50-mark threshold":
            int(((df["exam_score"] >= 50).astype(int) != df["target"]).sum()),
        "Pass rate": round(df["target"].mean(), 2),
    } for name, df in data.items()
})
quality = quality.apply(lambda col: col.map(lambda v: f"{v:g}"))  

In [39]:
averages = pd.DataFrame({n: data[n][NUMERIC + ["exam_score"]].mean().round(1) for n in names})

In [41]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
rows = []
for name, df in data.items():
    for kind in ("logistic", "forest"):
        res = cross_validate(make_model(kind), df[FEATURES], df["target"], cv=cv,
                             scoring=["accuracy", "f1", "roc_auc"])
        row = {"dataset": name, "model": kind}
        for m in ("accuracy", "f1", "roc_auc"):
            row[m + "_mean"] = round(res["test_" + m].mean(), 3)
            row[m + "_std"] = round(res["test_" + m].std(), 3)
        rows.append(row)
cross_validation = pd.DataFrame(rows)

In [43]:
rows = []
for a, b in [(names[0], names[1]), (names[1], names[0])]:
    model = make_model("logistic").fit(data[a][FEATURES], data[a]["target"])
    pred = model.predict(data[b][FEATURES])
    proba = model.predict_proba(data[b][FEATURES])[:, 1]
    rows.append({"train_on": a, "test_on": b,
                 "accuracy": round(accuracy_score(data[b]["target"], pred), 3),
                 "f1": round(f1_score(data[b]["target"], pred), 3),
                 "roc_auc": round(roc_auc_score(data[b]["target"], proba), 3)})
cross_test = pd.DataFrame(rows)

In [45]:
rows = []
for name, df in data.items():
    model = make_model("logistic").fit(df[FEATURES], df["target"])
    labels = model.named_steps["prep"].get_feature_names_out()
    coefs = model.named_steps["clf"].coef_[0]
    for i in np.argsort(-np.abs(coefs)):
        rows.append({"dataset": name, "feature": labels[i], "coefficient": round(coefs[i], 3)})
influence = pd.DataFrame(rows)

In [47]:
quality.to_csv(out / "quality_checks.csv")
averages.to_csv(out / "averages.csv")
cross_validation.to_csv(out / "cross_validation.csv", index=False)
cross_test.to_csv(out / "cross_test.csv", index=False)
influence.to_csv(out / "feature_influence.csv", index=False)

report = "\n\n".join([
    "SYNTHETIC DATASET EVALUATION REPORT",
    "1. QUALITY CHECKS\n" + quality.to_string(),
    "2. AVERAGE VALUES SIDE BY SIDE\n" + averages.to_string(),
    "3. USEFULNESS (5-fold cross-validation, mean and spread)\n" + cross_validation.to_string(index=False),
    "4. CROSS-TEST\n" + cross_test.to_string(index=False),
    "5. FEATURE INFLUENCE (positive = raises the chance of passing)\n" + influence.to_string(index=False),
])
(out / "evaluation_report.txt").write_text(report, encoding="utf-8")

print(f"Saved to the '{OUT_FOLDER}' folder:")
for p in sorted(out.iterdir()):
    print("  ", p.name)

Saved to the 'results' folder:
   averages.csv
   cross_test.csv
   cross_validation.csv
   evaluation_report.txt
   feature_influence.csv
   quality_checks.csv
